In [ ]:
import gc
import os
import sys
from pathlib import Path
import jax
jax.config.update("jax_traceback_filtering", "off")
jax.config.update("jax_compilation_cache_dir", "../jax-caches")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
import numpy as np
from scipy.optimize import minimize
from desc.dipole import _Dipole, DipoleSet, import_dipoles, export_dipoles
from desc.io import load
from desc.grid import LinearGrid, Grid
from desc.magnetic_fields._core import dipole_field
from desc.objectives import (ObjectiveFunction, FixParameters, DipoleDiscreteness, DipoleVolume, QuadraticFluxPM, DipoleTikhonov)
from desc.objectives.linear_objectives import FixDipoleRho
from desc.optimize import Optimizer
import matplotlib.pyplot as plt
import jax.numpy as jnp
from desc.magnetic_fields import SumMagneticField, SplineMagneticField
from desc.compute import get_params
from desc.equilibrium import Equilibrium
from desc.plotting import plot_comparison, plot_section

eq = load('qipm3.h5')
coilset = load('qipm_coils.h5')
one_period = import_dipoles(3, True, 'qipm_dipoles.csv')

coil_grid   = LinearGrid(N=40)
plasma_grid = LinearGrid(M=8, N=30, NFP=eq.NFP, sym=eq.sym)

weights = {"quadratic flux": 1000,"discreteness": 1,"volume": 5}

n_dipoles = len(one_period)
dipoles = DipoleSet(*[one_period[i] for i in range(n_dipoles)],NFP=3,sym=True)
optimizer = Optimizer("scipy-l-bfgs-b")

obj = ObjectiveFunction(
        (
            QuadraticFluxPM(
                eq, field=dipoles, eval_grid=plasma_grid, field_fixed=coilset,
                vacuum=True, weight=weights["quadratic flux"], bs_chunk_size=50,
            ),
            # DipoleDiscreteness(
            #     dipole=dipoles, weight=weights["discreteness"], 
            # ),
            # DipoleVolume(
            #     dipole=dipoles, weight=weights["volume"], 
            # )
        )
     )

(opt_d,), _ = optimizer.optimize(
    dipoles,
    objective=obj,
    maxiter=500,
    verbose=3,
    ftol=1e-8,
    copy=True,
)


Building objective: Quadratic flux
Precomputing transforms
Using precomputed induction matrix for 1 fixed-geometry dipole field(s)
Timer: Precomputing transforms = 8.01 sec
Timer: Objective build = 31.7 sec
Number of parameters: 35620
Number of objectives: 549
Timer: Initializing the optimization = 33.9 sec

Starting optimization
Using method: scipy-l-bfgs-b
   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1          5.206e+06                                    8.895e+03   
       1              3          3.920e+06      1.286e+06      5.000e+00      6.576e+03   
       2              4          8.912e+05      3.029e+06      2.664e+01      2.271e+04   
       3              5          7.431e+05      1.480e+05      1.251e+01      5.133e+04   
       4              6          4.111e+05      3.320e+05      3.479e+00      3.989e+04   
       5              7          2.935e+05      1.176e+05      5.217e+00      9.574e+03   
  

In [ ]:
# calculate max axis-outer dipole distance
# import numpy as np
from scipy.spatial import cKDTree
from desc.grid import LinearGrid

# Get the magnetic axis from DESC
axis_grid = LinearGrid(
    rho=[0.0],
    M=0,
    zeta=np.linspace(0, np.pi / eq.NFP, 4096),
    NFP=eq.NFP,
)
axis_data = eq.axis.compute(["X", "Y", "Z"], grid=axis_grid)

axis_xyz = np.column_stack([
    np.asarray(axis_data["X"]).ravel(),
    np.asarray(axis_data["Y"]).ravel(),
    np.asarray(axis_data["Z"]).ravel(),
])

mags_xyz = np.column_stack((one_period.X, one_period.Y, one_period.Z))



# Identify the outermost radial layer among active magnets.
# Assumes the focus file stores magnets ordered by tower and then layer.
# If not, use the layer indices from the grid generator instead.
tree_axis = cKDTree(axis_xyz)
distances, _ = tree_axis.query(mags_xyz, k=1)

# Approximate outermost layer as the magnets with the largest axis distance.
# This is a convenient estimate, but see note below.
outer_radius_est = distances.max()
tol = 1e-5  # m
outer_mask = distances >= outer_radius_est - tol

outer_distances = distances[outer_mask]

print(f"Maximum axis-to-dipole-center distance: {outer_distances.max():.6f} m")
print(f"Maximum axis-to-dipole-center distance: {outer_distances.max()*100:.3f} cm")
print(f"Maximum axis-to-dipole-center distance: {outer_distances.max()*1000:.3f} mm")
print(f"Magnets selected in outermost-radius band: {outer_mask.sum()}")

In [ ]:
import numpy as np
import plotly.graph_objects as go

from desc.grid import LinearGrid
from desc.magnetic_fields import SumMagneticField

# --------------------------------------------------
# 1. Combined magnetic field
# --------------------------------------------------
field = SumMagneticField(coilset, opt_d)

# --------------------------------------------------
# 2. Equilibrium surface grid: ONE field period
# --------------------------------------------------
M = 40
N = 80

eq_grid = LinearGrid(
    rho=np.array([1.0]),
    M=M,
    N=N,
    NFP=eq.NFP,
    sym=False,
    endpoint=True,  # close the theta and zeta boundaries
)

eq_data = eq.compute(
    ["X", "Y", "Z", "R", "n_rho"],
    grid=eq_grid,
)

# IMPORTANT:
# DESC uses theta-fastest ordering.
# order="F" gives shape (n_theta, n_zeta).
n_theta = eq_grid.num_theta
n_zeta = eq_grid.num_zeta

def surface_reshape(arr):
    return np.asarray(arr).reshape(
        (n_theta, n_zeta), order="F"
    )

X_eq = surface_reshape(eq_data["X"])
Y_eq = surface_reshape(eq_data["Y"])
Z_eq = surface_reshape(eq_data["Z"])

# --------------------------------------------------
# 3. Evaluate B dot n on the equilibrium surface
# --------------------------------------------------
# Magnetic field evaluation coordinates are
# [R, phi, Z], with phi = zeta.

pts_rpz = np.column_stack([
    np.asarray(eq_data["R"]),
    np.asarray(eq_grid.nodes[:, 2]),
    np.asarray(eq_data["Z"]),
])

B = np.asarray(
    field.compute_magnetic_field(
        pts_rpz,
        basis="rpz",
    )
)

# DESC n_rho is an area-weighted surface normal,
# expressed in cylindrical coordinates by default.
n_rho = np.asarray(eq_data["n_rho"])

Bdotn = np.sum(B * n_rho, axis=-1)
Bdotn_surface = surface_reshape(Bdotn)

# Symmetric color range
vmax = np.nanmax(np.abs(Bdotn_surface))
if vmax == 0:
    vmax = 1.0

print("Max |B dot n|:", np.nanmax(np.abs(Bdotn)))
print("RMS B dot n:", np.sqrt(np.nanmean(Bdotn**2)))

# --------------------------------------------------
# 4. Plot equilibrium with B dot n contours
# --------------------------------------------------
fig = go.Figure()

fig.add_trace(go.Surface(
    x=X_eq,
    y=Y_eq,
    z=Z_eq,

    surfacecolor=Bdotn_surface,

    colorscale="RdBu_r",
    cmin=-vmax,
    cmax=vmax,
    cmid=0,

    colorbar=dict(
        title="B·n (T m²)",
        thickness=18,
        len=0.7,
        x=1.02,
    ),

    opacity=0.92,
    showscale=True,
    name="Equilibrium B·n",
    hovertemplate=(
        "X: %{x:.4f} m<br>"
        "Y: %{y:.4f} m<br>"
        "Z: %{z:.4f} m<br>"
        "B·n: %{surfacecolor:.3e}<extra></extra>"
    ),
))

# --------------------------------------------------
# 5. Dipole locations
# --------------------------------------------------
# dip_xyz = np.column_stack([
#     np.asarray(one_period.X).ravel(),
#     np.asarray(one_period.Y).ravel(),
#     np.asarray(one_period.Z).ravel(),
# ])

# fig.add_trace(go.Scatter3d(
#     x=dip_xyz[:, 0],
#     y=dip_xyz[:, 1],
#     z=dip_xyz[:, 2],

#     mode="markers",

#     marker=dict(
#         size=2,
#         color="red",
#         opacity=0.8,
#     ),

#     name="Dipoles",
# ))

# --------------------------------------------------
# 6. Coilset
# --------------------------------------------------
# coil_grid = LinearGrid(N=100)

# def flatten_coils(obj):
#     if hasattr(obj, "coils"):
#         return [
#             c
#             for sub in obj.coils
#             for c in flatten_coils(sub)
#         ]
#     if hasattr(obj, "__iter__") and not hasattr(obj, "compute"):
#         return [
#             c
#             for sub in obj
#             for c in flatten_coils(sub)
#         ]
#     return [obj]

# for i, coil in enumerate(flatten_coils(coilset)):
#     data = coil.compute(
#         ["X", "Y", "Z"],
#         grid=coil_grid,
#         basis="xyz",
#     )

#     fig.add_trace(go.Scatter3d(
#         x=np.asarray(data["X"]).ravel(),
#         y=np.asarray(data["Y"]).ravel(),
#         z=np.asarray(data["Z"]).ravel(),

#         mode="lines",
#         line=dict(color="blue", width=4),

#         name="Coils" if i == 0 else f"Coil {i+1}",
#         showlegend=(i == 0),
#     ))

# --------------------------------------------------
# 7. Layout
# --------------------------------------------------
fig.update_layout(
    title="Equilibrium Surface: Normal Field Error, Coils and Dipoles",

    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)",
        aspectmode="data",
    ),

    width=1100,
    height=850,

    legend=dict(
        x=0.02,
        y=0.98,
    ),
)

fig.show()

In [ ]:
export_dipoles(opt_d, 'qipm0')

In [ ]:
opt_d = import_dipoles(3, True, 'qipm0')

In [ ]:
for d in opt_d3:
    if d.rho_tilde<=-0.5:
        d.rho_tilde=-1
    elif -0.5<d.rho_tilde<0.5:
        d.rho_tilde=0
    else:
        d.rho_tilde=1

In [ ]:
#calculate bn errors
continuous_field = SumMagneticField(
    opt_d,
    coilset,
)
grid = LinearGrid(
    rho=np.array([1.0]),
    M=8,
    N=10,
    NFP=eq.NFP,
    sym=eq.sym,
)
Bn1, coords = continuous_field.compute_Bnormal(
    eq.surface,
    eval_grid=grid,
)
print(np.sum(Bn1**2))

In [ ]:
dipset=opt_d
rho_tilde = np.tanh(np.asarray(dipset.rho))

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.hist(rho_tilde, bins=100)
ax.set_xlabel("rho_tilde")
ax.set_title(f"||rho_tilde||={np.linalg.norm(rho_tilde):.3e}, ")

plt.tight_layout()
plt.show()

print(f"mean={rho_tilde.mean():.4e}  std={rho_tilde.std():.4e}  "
      f"min={rho_tilde.min():.4e}  max={rho_tilde.max():.4e}")

magfield = SumMagneticField(
    dipset,
    coilset,
)
grid = LinearGrid(rho=np.array([1.0]),M=8, N=10,NFP=eq.NFP,sym=eq.sym,)
Bn1, coords = magfield.compute_Bnormal(eq.surface,eval_grid=grid)

print("bnorm:",np.sum(Bn1**2))
print("pmvol:",np.sum(rho_tilde**2))
print("dpbin:",jnp.mean(jnp.abs(rho_tilde) * (1 - jnp.abs(rho_tilde))))

In [ ]:
JAX_DISABLE_JIT=1

In [ ]:
#poincare plots using field_line_trace
from desc.plotting import plot_surfaces
from desc.magnetic_fields import field_line_integrate

field = SumMagneticField(coilset, opt_d)

rho_start = np.linspace(0.1, 0.95, 8)
grid_trace = LinearGrid(rho=rho_start)
r0 = np.asarray(eq.compute("R", grid=grid_trace)["R"])
z0 = np.asarray(eq.compute("Z", grid=grid_trace)["Z"])

# r0_in = np.asarray(eq.compute("R", grid=grid_trace)["R"])
# z0_in = np.asarray(eq.compute("Z", grid=grid_trace)["Z"])

# # axis (rho=0) and LCFS (rho=1) at theta=zeta=0, to get an outward direction
# grid_axis = LinearGrid(rho=np.array([0.0]))
# grid_lcfs = LinearGrid(rho=np.array([1.0]))
# r_axis = np.asarray(eq.compute("R", grid=grid_axis)["R"])[0]
# z_axis = np.asarray(eq.compute("Z", grid=grid_axis)["Z"])[0]
# r_lcfs = np.asarray(eq.compute("R", grid=grid_lcfs)["R"])[0]
# z_lcfs = np.asarray(eq.compute("Z", grid=grid_lcfs)["Z"])[0]

# # extrapolate a few points beyond the LCFS along the axis->LCFS direction
# extrap_factors = np.array([1.1, 1.2, 1.3])   # 5%, 15%, 30%, 50% further out than LCFS
# r0_out = r_axis + extrap_factors * (r_lcfs - r_axis)
# z0_out = z_axis + extrap_factors * (z_lcfs - z_axis)

# r0 = np.concatenate([r0_in, r0_out])
# z0 = np.concatenate([z0_in, z0_out])

ntransit = 2
phi = np.linspace(0, 2 * np.pi / eq.NFP, 3, endpoint=False)
#phi=np.array([0,np.pi/4,np.pi/2])
nplanes = len(phi)
phis = (phi + np.arange(0, ntransit)[:, None] * 2 * np.pi / eq.NFP).flatten()

fieldR, fieldZ, aux = field_line_integrate(
    r0, z0, phis, field,
    bounds_R=(0.1, 0.6),
    bounds_Z=(-0.3, 0.3),
    max_steps=10000000,
    options={"throw": True},
    return_aux=True,
)

print(aux)

R = np.asarray(fieldR).reshape((ntransit, nplanes, -1))
Z = np.asarray(fieldZ).reshape((ntransit, nplanes, -1))

rows = int(np.floor(np.sqrt(nplanes)))
cols = int(np.ceil(nplanes / rows))
fig, ax = plt.subplots(1, 3, figsize=(4 * cols, 5 * rows), squeeze=False,
                       sharex=True, sharey=True, subplot_kw=dict(aspect="equal"))
for i in range(nplanes):
    ax.flat[i].scatter(R[:, i, :], Z[:, i, :], color="tab:blue", s=0.5)
    ax.flat[i].set_xlabel("R")
    ax.flat[i].set_ylabel("Z")
    ax.flat[i].set_title(f"phi*NFP/2pi = {eq.NFP * phi[i] / (2 * np.pi):.3f}")


plot_surfaces(eq, rho=np.array([1.0]), theta=1, phi=phi, ax=ax)

for a in ax.flat:
    a.tick_params(labelbottom=True, labelleft=True)

grid_bn = LinearGrid(rho=np.array([1.0]), theta=100, zeta=phi, NFP=eq.NFP)
data_bn = eq.compute(["R", "Z", "n_rho"], grid=grid_bn)
pts_rpz = np.stack([data_bn["R"], grid_bn.nodes[:, 2], data_bn["Z"]], axis=-1)

B = field.compute_magnetic_field(pts_rpz, basis="rpz")
Bdotn = np.sum(B * data_bn["n_rho"], axis=-1)


vmax = np.max(np.abs(Bdotn))
for i, ph in enumerate(phi):
    mask = np.isclose(grid_bn.nodes[:, 2], ph)
    sc = ax.flat[i].scatter(
        data_bn["R"][mask], data_bn["Z"][mask], c=Bdotn[mask],
        cmap="RdBu_r", vmin=-vmax, vmax=vmax, s=10, zorder=5,
    )
fig.colorbar(sc, ax=ax, label="B·n residual (T·m²)", shrink=0.6)
fig.tight_layout()
#fig.savefig("poincare_vs_target_with_bdotn.png", dpi=200)
plt.show()


# from IPython.display import display
# display(fig)


In [ ]:
idx = 6

for turns in [1, 1.25, 1.5, 1.75, 2, 2.5, 3]:
    phis_test = np.linspace(
        0,
        turns * 2*np.pi/eq.NFP,
        30,
    )

    Rt, Zt, (stats, result) = field_line_integrate(
        r0[idx:idx+1],
        z0[idx:idx+1],
        phis_test,
        field,
        bounds_R=(0.1, 0.6),
        bounds_Z=(-0.3, 0.3),
        max_steps=300000,
        options={"throw": False},
        return_aux=True,
    )

    print(
        f"Turns={turns:.2f}, "
        f"steps={np.asarray(stats['num_steps'])[0]}, "
        f"result={result}"
    )

In [ ]:
R_bad = np.asarray(R_bad)[:, 0]
Z_bad = np.asarray(Z_bad)[:, 0]

coords = np.stack(
    [R_bad, phis, Z_bad],
    axis=-1
)

B = np.asarray(
    field.compute_magnetic_field(coords, basis="rpz")
)

Bphi = B[:, 1]

print("Minimum |Bphi|:", np.min(np.abs(Bphi)))
print("Bphi values:", Bphi)
print("R range:", R_bad.min(), R_bad.max())
print("Z range:", Z_bad.min(), Z_bad.max())

In [ ]:
#plot cross section of vmec with dipoles in that section
import numpy as np
import matplotlib.pyplot as plt
from desc.grid import LinearGrid


X_dip = np.array([float(d.X) for d in one_period])
Y_dip = np.array([float(d.Y) for d in one_period])

grid_phi0 = LinearGrid(rho=1.0, theta=200, zeta=np.array([0.0, np.pi/4, np.pi/2]), NFP=1, endpoint=True)
data_phi0 = eq.compute(["R", "Z"], grid=grid_phi0)

phi_dip = np.arctan2(Y_dip, X_dip)
near_phi0 = np.abs(phi_dip) < 0.05   # radians -- widen/narrow depending on dipole spacing

data_phi0 = eq.compute(["R", "Z"], grid=grid_phi0)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(data_phi0["R"], data_phi0["Z"], color="C0")
ax.scatter(R_dip[near_phi0], Z_dip[near_phi0], s=10, color="red")
ax.set_xlabel("R (m)")
ax.set_ylabel("Z (m)")
ax.set_aspect("equal")
ax.grid(True, linestyle="--", alpha=0.4)
plt.show()

In [ ]:
# calculation of Bn the normal way
grid = LinearGrid(M=8, N=12, NFP=eq.NFP, sym=False, endpoint=True)  # calc_g's default grid
data = eq.compute(["X", "Y", "Z"], grid=grid)
x = jnp.stack([data["X"], data["Y"], data["Z"]], axis=-1)
n_surf = eq.surface.compute(["n_rho"], grid=grid, basis="xyz")["n_rho"]

B = one_period.compute_magnetic_field(x, basis="xyz")
Bn = jnp.sum(B * n_surf, axis=-1)

In [ ]:
# use inductance matrix to calculate Bn
g_ij = one_period.calc_g(eq)
M0 = one_period.m0 * jnp.tanh(jnp.asarray(one_period.rho))
Bn_via_g = g_ij @ M0